# Experiment Benefit of Depth
In this notebook the benefit of depth is tested on a CNN base model that was established in the notebook "CNN-architecture-testing".

### Resaerch Question:
How does the depth of the neural network influence the model accuracy and loss?

### Expectation:
It is expected that the model accuracy will increase and the model loss will decrease respectively.\
But also the training time is expected to increase with more layers.\
It is thought that adding more hidden layers will have stagnating improvement untill to the point where there is no more improvement.

### Framework:
CNN with 

In [ ]:
class shallow_model(nn.Module):
    
    def __init__(self, units=128, drop=0.5):
        super(shallow_model, self).__init__()
        self.seq = nn.Sequential(
            # Layer 1---------------------------------------------------------------------------------------
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # Conv with 32 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 224x224 -> 112x112

            # Layer 2---------------------------------------------------------------------------------------
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # Conv with 64 filters, kernel size 3x3, padding 1
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2), # 112x112 -> 56x56

            # output layer----------------------------------------------------------------------------------
            nn.Flatten(),
            nn.Linear(56*56*64,units),
            nn.ReLU(),
            nn.Dropout(drop), # dropout rate of 0.5 as deafault value
            nn.Linear(units,10) # output layer with 10 units for 10 classes
        )
        
    
    def forward(self, x):
        return self.seq(x)

In [ ]:
import copy
import time
from dataclasses import dataclass, asdict
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import wandb


# =========================================================
# CONFIG
# =========================================================

@dataclass # using dataclass for cleaner configuration management, allows easy instantiation and conversion to dict for W&B logging
# Settings for model architecture, training hyperparameters, and W&B logging
class Config:
    in_channels: int = 3
    num_classes: int = 10

    depths: Tuple[int, ...] = (1, 2, 4, 8, 16)

    batch_size: int = 64
    epochs: int = 50
    lr: float = 1e-3
    weight_decay: float = 1e-4

    base_channels: int = 32
    max_channels: int = 256
    dropout: float = 0.0 # dropout for confolutional blocks, can be set to zero if not needed since it was found to not help in this setting and just increases training time, but can be easily tuned if needed
    do: float = 0.5 # dropout rate for the final classifier head, can be tuned as a hyperparameter if needed

    num_workers: int = 0 # set to 0 for Windows to avoid issues with multiprocessing, can be set to >0 for faster data loading on Linux/Mac
    pin_memory: bool = True # pin_memory=True can speed up data transfer to GPU, generally recommended when using a GPU

    device: str = "cuda" if torch.cuda.is_available() else "cpu" # automatically use GPU if available, otherwise fall back to CPU

    # W&B
    use_wandb: bool = True
    wandb_project: str = "MPW-CNN"
    wandb_entity: str = "MSE_DeLearn_SPR26"
    wandb_mode: str = "online" # "online", "offline", or "disabled"
    wandb_watch: bool = False 
    wandb_log_freq: int = 100


# =========================================================
# MODEL
# =========================================================

# A convolutional block with optional pooling and dropout, can be reused to build deeper CNNs
class ConvBlock(nn.Module):
    """
    A basic convolutional block consisting of Conv2d -> BatchNorm2d -> ReLU, with optional MaxPool2d and Dropout2d.
    """
    def __init__(self, in_channels: int, out_channels: int, use_pool: bool = False, dropout: float = 0.0):
        super().__init__()

        layers = [
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding='same', bias=False), # conv with kernel size 3x3, padding 'same', no bias (since we have batch norm)
            nn.BatchNorm2d(out_channels), # batch normalization for better training stability
            nn.ReLU(inplace=True), # ReLU activation with inplace=True for memory efficiency
        ]

        if use_pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2)) # max pooling with kernel size 2x2 and stride 2 to downsample the feature maps

        if dropout > 0:
            layers.append(nn.Dropout2d(dropout)) # spatial dropout to randomly zero out entire channels during training for regularization

        self.block = nn.Sequential(*layers) # sequential container to hold the layers of the block

    def forward(self, x):
        return self.block(x)


# Construction of the main CNN model with variable depth using the ConvBlock defined above
class DepthCNN(nn.Module):
    """
    CNN with configurable number of convolutional layers.
    Supported depths: 1, 2, 4, 8, 16
    """

    def __init__(
        self,
        depth: int,
        in_channels: int = 3,
        num_classes: int = 10,
        base_channels: int = 32, # number of filters in the first conv layer, will be doubled after every second conv layer
        max_channels: int = 256, # maximum number of filters to prevent excessive model size
        dropout: float = 0.0, # dropout reate set to zero after discussion with Jean.
                              # He has found that dropout does not help in this setting and just increases training time, so we will set it to zero by default
        do: float = 0.5, # dropout rate for the final classifier head
    ):
        super().__init__()

        if depth not in {1, 2, 4, 8, 16}: # only support specific depths to keep the model sizes reasonable and comparable
            raise ValueError(f"Depth must be one of {{1,2,4,8,16}}, got {depth}")

        layers = []

        current_in = in_channels
        current_out = base_channels

        # build the convolutional part of the model by stacking ConvBlocks according to the specified depth, doubling the number of filters after every second block until max_channels is reached
        for i in range(depth):
            use_pool = ((i + 1) % 2 == 0)  # pool after every second conv layer common practice to reduce spatial dimensions while increasing feature channels

            layers.append(
                ConvBlock(
                    in_channels=current_in, # number of input channels for this block, will be the output channels of the previous block
                    out_channels=current_out, # number of output channels for this block, will be doubled after every second block until max_channels is reached
                    use_pool=use_pool, # whether to apply max pooling in this block, applied after every second conv layer see above
                    dropout=dropout if depth >= 4 else 0.0, # is set to zero see above
                )
            )

            current_in = current_out # for the next block, the input channels will be the output channels of the current block

            if use_pool:
                current_out = min(current_out * 2, max_channels)

        self.features = nn.Sequential(*layers) # sequential container to hold all the convolutional blocks as the feature extractor part of the model

        # Final classifier head that takes the output of the convolutional part and produces class logits, using adaptive average pooling to handle variable spatial dimensions and a fully connected layer to produce the final class scores
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)), # global average pooling to reduce the spatial dimensions to 1x1, resulting in a feature vector of size current_in
            nn.Flatten(),
            nn.Dropout(p=do), # dropout before the final linear layer for regularization
            nn.Linear(current_in, num_classes)
        )

    # forward pass through the model: first through the convolutional feature extractor, then through the classifier head to produce the final logits
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


# =========================================================
# UTILITIES
# =========================================================

# counts the total number of trainable parameters in the model by summing the number of elements in each parameter tensor that requires gradients
def get_num_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad) 

# not used since dataset is given with fixed evenly distributed numbers of samples per class
def count_dataset_classes(dataset, num_classes: int) -> Dict[int, int]:
    """
    Tries to count class frequencies for common PyTorch dataset structures.
    Works for datasets exposing .targets or .labels.
    """
    counts = {i: 0 for i in range(num_classes)}

    if hasattr(dataset, "targets"):
        targets = dataset.targets
    elif hasattr(dataset, "labels"):
        targets = dataset.labels
    else:
        return counts

    if isinstance(targets, torch.Tensor):
        targets = targets.tolist()

    for y in targets:
        counts[int(y)] += 1

    return counts


# =========================================================
# TRAIN / EVAL FUNCTIONS
# =========================================================

# Function to train the model for one epoch, iterating over the training data loader, computing the loss and accuracy, and updating the model parameters using backpropagation and the optimizer.
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: torch.optim.Optimizer,
    device: torch.device
):
    model.train()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    start_time = time.time() # timimg the epoch to report average epoch time later

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True) # zero the gradients before the backward pass to prevent accumulation from previous batches

        logits = model(x) # forward pass to get the predicted logits for the input batch
        loss = criterion(logits, y) # compute the loss between the predicted logits and the true labels

        loss.backward() # compute gradients of the loss with respect to model parameters
        optimizer.step() # update model parameters based on the computed gradients

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    # compute average loss and accuracy for the epoch, and also calculate the time taken for the epoch
    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    epoch_time = time.time() - start_time

    return epoch_loss, epoch_acc, epoch_time


@torch.no_grad() # no_grad context since we are only evaluating and don't need gradients, which saves memory and computation
# Function to evaluate the model on a given data loader (e.g., validation or test set), computing the average loss and accuracy without updating the model parameters.
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device
):
    model.eval()

    running_loss = 0.0
    running_correct = 0
    running_total = 0

    for x, y in loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        batch_size = y.size(0)
        running_loss += loss.item() * batch_size
        running_correct += (logits.argmax(dim=1) == y).sum().item()
        running_total += batch_size

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total

    return epoch_loss, epoch_acc


# =========================================================
# TRAIN ONE MODEL WITH W&B
# =========================================================

# Function to train a single model with a specified depth and settings
def train_model(
    depth: int,
    train_dataset,
    val_dataset,
    cfg: Config
):
    device = torch.device(cfg.device)

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    model = DepthCNN(
        depth=depth,
        in_channels=cfg.in_channels,
        num_classes=cfg.num_classes,
        base_channels=cfg.base_channels,
        max_channels=cfg.max_channels,
        dropout=cfg.dropout,
        do=cfg.do,
    ).to(device)

    criterion = nn.CrossEntropyLoss() # standard cross-entropy loss for multi-class classification
    optimizer = torch.optim.Adam( # using Adam optimzer --> State of the art
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    history = {
        "train_loss": [],
        "train_acc": [],
        "val_loss": [],
        "val_acc": [],
        "epoch_time_sec": [],
    }

    best_val_acc = -1.0
    best_val_loss = float("inf")
    best_epoch = -1
    best_model_state = copy.deepcopy(model.state_dict())

    num_params = get_num_parameters(model)

    run = None
    # initiate W&B run 
    if cfg.use_wandb and cfg.wandb_mode != "disabled":
        wandb_config = asdict(cfg)
        wandb_config.update({
            "depth": depth,
            "num_parameters": num_params,
            "optimizer": "Adam",
            "loss": "CrossEntropyLoss",
        })

        run = wandb.init(
            project=cfg.wandb_project,
            entity=cfg.wandb_entity,
            mode=cfg.wandb_mode,
            name=f"cnn_depth_{depth}",
            config=wandb_config,
            reinit=True,
            settings=wandb.Settings(init_timeout=300),
        )

        if cfg.wandb_watch:
            wandb.watch(model, log="all", log_freq=cfg.wandb_log_freq)

    print("=" * 80)
    print(f"Training model with depth = {depth}")
    print(f"Trainable parameters: {num_params:,}")
    print(model) # print the model architecture for reference
    print("=" * 80)

    for epoch in range(cfg.epochs):
        train_loss, train_acc, epoch_time = train_one_epoch(
            model, train_loader, criterion, optimizer, device
        )
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, device
        )

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["epoch_time_sec"].append(epoch_time)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_val_loss = val_loss
            best_epoch = epoch + 1
            best_model_state = copy.deepcopy(model.state_dict())

        current_lr = optimizer.param_groups[0]["lr"]

        print(
            f"Epoch [{epoch+1:02d}/{cfg.epochs:02d}] | "
            f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
            f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
            f"time={epoch_time:.1f}s"
        )

        # log metrics to W&B if enabled, including the train/val gap for both accuracy and loss to monitor overfitting
        if run is not None:
            wandb.log({
                "epoch": epoch + 1,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
                "train_val_gap/accuracy": train_acc - val_acc,
                "train_val_gap/loss": val_loss - train_loss,
                "epoch_time_sec": epoch_time,
                "lr": current_lr,
                "best_val_accuracy_so_far": best_val_acc,
            })

    model.load_state_dict(best_model_state) # load the best model state based on validation accuracy for final evaluation and reporting


    best_train_loss, best_train_acc = evaluate(model, train_loader, criterion, device)
    best_val_loss_eval, best_val_acc_eval = evaluate(model, val_loader, criterion, device)

    result = {
        "depth": depth,
        "best_val_acc": best_val_acc,
        "best_val_loss": best_val_loss,
        "best_epoch": best_epoch,
        "best_train_acc": best_train_acc,
        "best_val_acc_eval": best_val_acc_eval,
        "avg_epoch_time_sec": sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"]),
        "num_parameters": num_params,
    }

    # result = {
    #     "depth": depth,
    #     "best_val_acc": best_val_acc,
    #     "best_val_loss": best_val_loss,
    #     "best_epoch": best_epoch,
    #     "final_train_acc": history["train_acc"][-1],
    #     "final_val_acc": history["val_acc"][-1],
    #     "avg_epoch_time_sec": sum(history["epoch_time_sec"]) / len(history["epoch_time_sec"]),
    #     "num_parameters": num_params,
    # }

    # log final results to W&B summary for easy comparison across runs
    if run is not None:
        wandb.summary["best_epoch"] = best_epoch
        wandb.summary["best_val_accuracy"] = best_val_acc
        wandb.summary["best_val_loss"] = best_val_loss
        wandb.summary["final_train_accuracy"] = history["train_acc"][-1]
        wandb.summary["final_val_accuracy"] = history["val_acc"][-1]
        wandb.summary["avg_epoch_time_sec"] = result["avg_epoch_time_sec"]
        wandb.finish()

    return model, history, result


# =========================================================
# TRAIN ALL DEPTH MODELS
# =========================================================

# Function to train models for all specified depths in the configuration, storing the trained models, their training histories, and final results in dictionaries for easy access and comparison.
def train_all_depth_models(
    train_dataset,
    val_dataset,
    cfg: Config
):
    all_models = {}
    all_histories = {}
    all_results = {}

    for depth in cfg.depths:
        model, history, result = train_model(
            depth=depth,
            train_dataset=train_dataset,
            val_dataset=val_dataset,
            cfg=cfg
        )

        all_models[depth] = copy.deepcopy(model).cpu() # store a copy of the trained model on CPU to save GPU memory, can be moved back to GPU for evaluation if needed
        all_histories[depth] = history
        all_results[depth] = result

    return all_models, all_histories, all_results


@torch.no_grad()
# Function to evaluate the best model on the test set, computing the final test loss and accuracy for reporting.
# Since there is no test set in the given dataset, this function is not used in the current implementation, but can be easily added if a test set becomes available.
def evaluate_test_set(model: nn.Module, test_dataset, cfg: Config):
    device = torch.device(cfg.device)

    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=cfg.pin_memory,
    )

    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)

    return {
        "test_loss": test_loss,
        "test_acc": test_acc
    }


# =========================================================
# MAIN EXECUTION
# =========================================================


cfg = Config(
    in_channels=3,
    num_classes=10,
    depths=(1, 2, 4, 8, 16),
    batch_size=64,
    epochs=30,
    lr=1e-3,
    weight_decay=1e-4,
    base_channels=32,
    max_channels=256,
    dropout=0.0, # set to zero for Dropout in ConvBlocks since it was found to not help in this setting and just increases training time, but can be easily tuned if needed
    do=0.5, # dropout rate for the final classifier head, can be tuned as a hyperparameter if needed
    num_workers=0,
    pin_memory=True,
    use_wandb=True,
    wandb_project="MPW-CNN",
    wandb_entity="MSE_DeLearn_SPR26",   # set to None if not needed
    wandb_mode="online",                # "offline" if online mode fails
    wandb_watch=False,                  # safer on Windows / VS Code notebooks
)

# train models for all specified depths and collect their results
all_models, all_histories, all_results = train_all_depth_models(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    cfg=cfg
)

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)

for depth, result in all_results.items():
    print(
        f"Depth {depth:>2} | "
        f"params={result['num_parameters']:,} | "
        f"best_val_acc={result['best_val_acc']:.4f} | "
        f"best_val_loss={result['best_val_loss']:.4f} | "
        f"best_epoch={result['best_epoch']} | "
        f"final_train_acc={result['final_train_acc']:.4f} | "
        f"final_val_acc={result['final_val_acc']:.4f} | "
        f"avg_epoch_time={result['avg_epoch_time_sec']:.2f}s"
    )

# identify the best model based on validation accuracy and report its performance, optionally evaluate on the test set if available
best_depth = max(all_results.keys(), key=lambda d: all_results[d]["best_val_acc"])
best_model = all_models[best_depth]

print("\nBest model based on validation accuracy:")
print(f"Depth = {best_depth}")
print(f"Best validation accuracy = {all_results[best_depth]['best_val_acc']:.4f}")

# Optional test evaluation
# test_metrics = evaluate_test_set(best_model, test_dataset, cfg)
# print("\nTest set performance:")
# print(test_metrics)

# Optional: log summary of best model as separate W&B run --> stores a summary table comparing all depths and highlights the best one, useful for easy comparison and reporting in W&B dashboard
if cfg.use_wandb and cfg.wandb_mode != "disabled":
    run = wandb.init(
        project=cfg.wandb_project,
        entity=cfg.wandb_entity,
        mode=cfg.wandb_mode,
        name="cnn_depth_comparison_summary",
        reinit=True,
        settings=wandb.Settings(init_timeout=300),
    )

    comparison_table = wandb.Table(columns=[
        "depth",
        "num_parameters",
        "best_val_acc",
        "best_val_loss",
        "best_epoch",
        "final_train_acc",
        "final_val_acc",
        "avg_epoch_time_sec",
    ])

    for depth, result in all_results.items():
        comparison_table.add_data(
            result["depth"],
            result["num_parameters"],
            result["best_val_acc"],
            result["best_val_loss"],
            result["best_epoch"],
            result["final_train_acc"],
            result["final_val_acc"],
            result["avg_epoch_time_sec"],
        )

    wandb.log({
        "depth_comparison_table": comparison_table,
        "best_depth": best_depth,
        "best_depth_val_acc": all_results[best_depth]["best_val_acc"],
    })

    wandb.summary["best_depth"] = best_depth
    wandb.summary["best_depth_val_acc"] = all_results[best_depth]["best_val_acc"]
    wandb.finish()